# Step 1 - Ingest pdf file and generate markdown files

In [ ]:
%pip install openai
%pip install python-dotenv
%pip install pydub
%pip install azure-core
%pip install azure-cognitiveservices-speech
%pip install azure-ai-documentintelligence

In [ ]:
# Add azure open ai endpoint
import os,re
from dotenv import load_dotenv

load_dotenv(dotenv_path='../keys.env')
azure_openai_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')
azure_openai_key = os.getenv('AZURE_OPENAI_KEY')
azure_document_intelligence_endpoint = os.getenv('AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT')
azure_document_intelligence_key = os.getenv('AZURE_DOCUMENT_INTELLIGENCE_KEY')

folder_path = '../data/hachette'  # Adjust this path as needed

In [ ]:
# Specific settings for the books

pages2Process = {
    "A New Theory of Teenagers_ Seven Transform - Christa Santangelo": "8-125",
    "Screen Time - Lisa Guernsey": "8-316",
}

mdFileNameAddition = "_full_text"

In [ ]:
# Convert PDFs to MDs

from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentContentFormat
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI
from azure.core.exceptions import ClientAuthenticationError
from pydub import AudioSegment

# Replace with your Document Intelligence endpoint and key
aoai_endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
aoai_key = os.environ['AZURE_OPENAI_KEY']

# Replace with your Document Intelligence endpoint and key
doc_endpoint = os.environ['AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT']
doc_key = os.environ['AZURE_DOCUMENT_INTELLIGENCE_KEY']


def analyze_pdf(pdf_path, directory, filename):
    """
    Analyzes a PDF document using Azure Document Intelligence and returns the text,
    along with page-level information.
    """

    try:
        document_intelligence_client = DocumentIntelligenceClient(
            endpoint=doc_endpoint, credential=AzureKeyCredential(doc_key)
        )
        with open(pdf_path, "rb") as f:
            document_content = f.read()
            poller = document_intelligence_client.begin_analyze_document(
                "prebuilt-layout",
                output_content_format=DocumentContentFormat.MARKDOWN,
                pages=pages2Process[os.path.splitext(filename)[0]],
                body=document_content
            )
        result = poller.result()
        markdown_output = result.content

        full_text_file_path = os.path.join(directory, f"{os.path.splitext(filename)[0]}{mdFileNameAddition}.md")
        with open(full_text_file_path, "w", encoding="utf-8") as full_text_file:
            full_text_file.write(markdown_output)
        print(f"Full text written to {full_text_file_path}")

        return True, full_text_file_path

    except ClientAuthenticationError as e:
        print(f"Authentication error: {e}")
        print("Please check your Document Intelligence endpoint and key.")
        return None, None
    except Exception as e:
        print(f"Error analyzing {pdf_path}: {e}")
        return None, None

def process_directory(directory):
    """
    Processes all PDF files in the given directory.
    """
    for filename in os.listdir(directory):
        if filename.endswith(".pdf"):
            pdf_path = os.path.join(directory, filename)
            # Analyze PDF and get full_text_file_path
            full_text_process, full_text_file_path = analyze_pdf(pdf_path, directory, filename)

            if full_text_process == True:
                input_file = full_text_file_path
                # Split the Markdown file by chapters
                output_dir = os.path.join(directory, os.path.splitext(filename)[0]+'_chapters')

            else:
                print(f"Failed to extract text from {pdf_path}")

# launch the process
if __name__ == "__main__":
    # Define the root directory to scan
    root_directory = folder_path  # Use the folder_path variable
    # Iterate through all subdirectories in the root directory
    for subdir, dirs, files in os.walk(root_directory):
        print(f"Processing directory: {subdir}")
        process_directory(subdir)

In [ ]:
# Specific settings for detecting chapters for the books

chaptersRegex = {
    "A New Theory of Teenagers_ Seven Transform - Christa Santangelo": r'(\n\n\n#{1,2}\s*TRANSFORMATIVE STRATEGY #\d:\s*.*\n|# INTRODUCTION\n)',
    # "Screen Time - Lisa Guernsey": r'#{2,}\s*Chapter\b.*'
    "Screen Time - Lisa Guernsey": r'(#{2,3}\s*Chapter\s*.*\n\n\n#{1,}\s*.*\n|# Foreword to the Paperback Edition\n|# Preface, or the Three Cs: Content, Context and Your Child)'
}

In [ ]:
# Split MDs to Chapters MD

from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentContentFormat
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI
from azure.core.exceptions import ClientAuthenticationError
from pydub import AudioSegment


def split_markdown_by_chapters(input_file, output_dir):
    """
    Splits a Markdown file into separate chapter files based on chapter headings.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()

    # Split the content by chapter headings

    # Get filename without extension
    filenamewithoutext = os.path.splitext(os.path.basename(input_file))[0]
    # Remove end of string if equial to {mdFileNameAddition}
    filenamewithoutext = filenamewithoutext[:-len(mdFileNameAddition)] if filenamewithoutext.endswith(mdFileNameAddition) else filenamewithoutext 
    regexstr = chaptersRegex[filenamewithoutext]
    chapters = re.split(regexstr, content)
    
    # Ensure that the first item is not a chapter heading
    if chapters[0].strip() == '':
        chapters.pop(0)

    # Iterate through the chapters and write them to separate files
    for i in range(0, len(chapters), 2):
        chapter_heading = chapters[i].strip()
        chapter_content = chapters[i+1].strip() if i+1 < len(chapters) else ''
        # Create a valid filename from the chapter heading
        # Remove invalid characters from the chapter heading
        safe_chapter_heading = "".join(c if c.isalnum() or c in "._-" else "_" for c in chapter_heading)
        # Truncate the chapter heading to a reasonable length max 35 characters
        if len(safe_chapter_heading) > 35:
            safe_chapter_heading = safe_chapter_heading[:35]

        # Create a valid filename from the chapter heading
        chapter_filename = f"{i//2}_{safe_chapter_heading}.md"
        chapter_filepath = os.path.join(output_dir, chapter_filename)

        # Write the chapter content to the file
        with open(chapter_filepath, 'w', encoding='utf-8') as chapter_file:
            chapter_file.write(f"{chapter_heading}\n\n{chapter_content}")
        print(f"Chapter written to {chapter_filepath}")


def process_directory(directory):
    """
    Processes all MD files in the given directory.
    """
    for filename in os.listdir(directory):
        if filename.endswith("_full_text.md"):
            md_file = os.path.join(directory, filename)
              # Split the Markdown file by chapters
            output_dir = os.path.join(directory, os.path.splitext(filename)[0]+'_chapters')
            split_markdown_by_chapters(md_file, output_dir)

# launch the process
if __name__ == "__main__":
    # Define the root directory to scan
    root_directory = folder_path  # Use the folder_path variable
    # Iterate through all subdirectories in the root directory
    for subdir, dirs, files in os.walk(root_directory):
        print(f"Processing directory: {subdir}")
        process_directory(subdir)